## {PATTERN_NAME} Knowledge Graph Building Content

This notebook discovers documents in S3, extracts their text, and creates a Neo4j knowledge graph. It uses OpenShift AI Models-as-a-Service (MaaS) for embeddings and entity extraction. Run it before the graph inference notebook.

### &#x1F4CB; Contents 
This notebook contains the following sections:

- **[Setup](#Setup)**
  - [Disconnected cluster setup](#Disconnected-cluster-setup)
  - [Install packages](#Install-packages)
  - [Import required libraries](#Import-required-libraries)
  - [Configure S3 credentials](#Configure-S3-credentials)
  - [Prepare S3 client](#Prepare-S3-client)
- **[Process input documents](#Process-input-documents)**
  - [Documents discovery](#Documents-discovery)
  - [Prepare text extraction](#Prepare-text-extraction)
  - [Text extraction](#Text-extraction)
- **[Upload documents content into vector store](#Upload-documents-content-into-vector-store)**
  - [Prepare MaaS client](#Prepare-MaaS-client)
  - [Prepare chunker](#Prepare-chunker)
  - [Initialize vector store](#Initialize-vector-store)
  - [Upload chunks to vector store](#Upload-chunks-to-vector-store)
  - [Retrieve chunks for sample question](#Retrieve-chunks-for-sample-question)
- **[Summary](#Summary)**

---

## Setup

First choose the cluster mode. Then install ai4rag and connect to the S3 bucket that holds the documents and test data.

### Disconnected cluster setup

The default is a connected cluster. Before running on a disconnected cluster, complete these steps in your workbench environment:

1. Set `RHOAI_NOTEBOOK_CLUSTER_MODE=disconnected` (AutoX passes this as `NOTEBOOK_CLUSTER_MODE`).
2. Use a workbench image or internal package mirror with the generated ai4rag version, its `text-extraction` extra, and `nest-asyncio`. Keep a compatible OpenCV build available in that image or mirror.
3. For PDFs or images, mount the approved Docling bundle with its PP-OCRv4 files and set `DOCLING_ARTIFACTS_PATH`. For audio, mount the approved Whisper model and set `HF_MODEL_DIR`. Provide `ffmpeg` for audio. Text-only files need neither bundle.
4. Make S3, MaaS, and Neo4j reachable through internal endpoints. Add any private CA to the workbench trust store; for S3, you can set `AWS_CA_BUNDLE`.

See the [disconnected-cluster setup guide](https://github.com/IBM/ai4rag#running-on-disconnected-clusters) for artifact preparation. No environment-specific cells need to be skipped.

In [ ]:
import os

cluster_mode = (
    os.getenv("NOTEBOOK_CLUSTER_MODE")
    or os.getenv("RHOAI_NOTEBOOK_CLUSTER_MODE")
    or "connected"
).strip().lower()
if cluster_mode not in {{"connected", "disconnected"}}:
    raise ValueError("Notebook cluster mode must be 'connected' or 'disconnected'.")
if cluster_mode == "disconnected":
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
print(f"Notebook cluster mode: {{cluster_mode}}")

### Install packages

Install the ai4rag version used to generate this notebook. Connected runs use PyPI. On a disconnected cluster, configure an internal package mirror with this version before running the cell.

In [ ]:
import subprocess
import sys

install_command = [sys.executable, "-m", "pip", "install", "ai4rag[text-extraction]~={AI4RAG_VERSION}", "nest-asyncio"]
if cluster_mode == "connected":
    install_command.extend(["--index-url", "https://pypi.org/simple"])
subprocess.run(install_command, check=True)

### Import required libraries

Import all necessary Python modules and configure logging to suppress verbose output from component loggers.

In [ ]:
import getpass
import json
import logging
import nest_asyncio
import os
import warnings
from pathlib import Path

nest_asyncio.apply()

warnings.filterwarnings("ignore")

for logger_name in (
    "httpx",
    "documents-discovery",
    "text-extraction",
):
    logging.getLogger(logger_name).propagate = False

### Configure S3 credentials

To load documents from S3-compatible object storage, you need to provide credentials. If you're using OpenShift AI, these can be configured as data connections.

&#x1F4CC; **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

&#x1F4A1; **Tip**: In the project, open **Connections** and add an **S3 compatible object storage connection** to a bucket you will use for documents and test data. Open **Workbenches**, edit your workbench, and attach the S3 connection you created so the notebook can read from the bucket. Save and restart the workbench if prompted.

In [ ]:
required_vars = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_S3_BUCKET"]
missing = [var for var in required_vars if not os.environ.get(var)]
if missing:
    raise ValueError(f"Missing required environment variables: {{missing}}")

### Prepare S3 client

Creates an S3 client session using the provided credentials. This client will be used to discover and download documents from the specified S3 bucket.

In [ ]:
from ai4rag.utils.clients.s3 import create_s3_client

workbench_ca_bundle = Path("/etc/pki/tls/custom-certs/ca-bundle.crt")
ssl_cert_path = str(workbench_ca_bundle) if workbench_ca_bundle.is_file() else None
if ssl_cert_path:
    os.environ.setdefault("AWS_CA_BUNDLE", ssl_cert_path)

effective_ssl_cert_path = ssl_cert_path or os.getenv("AWS_CA_BUNDLE")
s3_client = create_s3_client(verify=effective_ssl_cert_path or True)
s3_client.head_bucket(Bucket=os.environ["AWS_S3_BUCKET"])
print("S3 connection verified with TLS certificate validation enabled.")

---

## Process input documents

This section handles document discovery and text extraction. Documents are first discovered in S3 storage, then their content is extracted and saved as DoclingDocument JSON files for further processing.

The data processing pipeline prepares documents for the RAG system in multiple steps. Each step produces outputs stored under `step_outputs/`. 

| Step | Function | Purpose |
|------|----------|---------| 
| 1 | **`discover_documents`** | List documents under every selected prefix, prioritize benchmark-referenced docs, apply a size cap, and write a JSON manifest (no content download). |
| 2 | **`extract_text`** | Download the listed documents from S3 and extract text to DoclingDocument JSON files using Docling. |

In [ ]:
from ai4rag.utils.data.documents_discovery import discover_documents
from ai4rag.utils.data.test_data_loader import load_test_data
from ai4rag.utils.data.text_extraction import extract_text, prepare_extraction

step_output_dir = Path("./step_outputs")
input_data_bucket_name = os.environ["AWS_S3_BUCKET"]
input_data_keys = {INPUT_DATA_KEYS}
test_data_key = "{TEST_DATA_KEY}"
step_output_dir.mkdir(parents=True, exist_ok=True)

### Documents discovery

Lists objects in the S3 input bucket, filters by supported extensions (e.g., `.pdf`, `.docx`, `.pptx`, `.md`, `.html`, `.txt`), and builds a document set. Documents referenced in the benchmark are prioritized, then others are added until a configurable size limit (1 GB by default) is reached. This step does not download document contents but writes a JSON manifest (`documents_descriptor.json`) containing the bucket, prefixes, and list of selected object keys and sizes for the next step.

In [ ]:
import inspect

test_data_result = load_test_data(
    bucket_name=input_data_bucket_name,
    key=test_data_key,
    s3_client=s3_client,
)
test_data_doc_names = [
    document_key
    for record in test_data_result.data
    for document_key in record["correct_answer_document_keys"]
]

discovery_kwargs = {{
    "bucket_name": input_data_bucket_name,
    "test_data_doc_names": test_data_doc_names,
    "s3_client": s3_client,
}}
if "prefixes" in inspect.signature(discover_documents).parameters:
    discovery_kwargs["prefixes"] = input_data_keys
else:
    if len(input_data_keys) > 1:
        raise RuntimeError(
            "This notebook was generated with multiple input locations, but the installed ai4rag "
            "does not support them. Install ai4rag >= 0.18.0."
        )
    discovery_kwargs["prefix"] = input_data_keys[0] if input_data_keys else ""

result = discover_documents(**discovery_kwargs)
result.save(step_output_dir / "discovered_documents")

print(json.dumps(result.to_dict(), indent=4, ensure_ascii=False))

### Prepare text extraction

Use the language selected by the experiment. The ai4rag helper checks which files need OCR or speech recognition. Connected runs download missing models; disconnected runs use the staged models described above.

In [ ]:
discovered_documents = result.to_dict()["documents"]
pattern_language = {LANGUAGE} or {{}}
extraction = prepare_extraction(
    discovered_documents,
    language=pattern_language.get("code") or pattern_language.get("name"),
    disconnected=cluster_mode == "disconnected",
)
print("Text extraction is ready.")

### Text extraction

Download the selected files from S3 and save extracted text as Docling JSON files. One worker keeps memory use suitable for a small workbench.

In [ ]:
extracted_text_dir = step_output_dir / "extracted_text"
extraction_result = extract_text(
    documents=discovered_documents,
    bucket=result.bucket,
    output_dir=extracted_text_dir,
    s3_client=s3_client,
    docling_artifacts_path=extraction.artifacts_path,
    docling_config=extraction.config,
    max_extraction_workers=1,
)

print(
    f"Extracted {{extraction_result.processed_count}}/{{extraction_result.total_documents}} documents "
    f"({{extraction_result.error_count}} errors)"
)

---

## Build the Neo4j knowledge graph

This section uses the documents extracted above to create a collection-scoped Neo4j knowledge graph.

### Prepare MaaS client

OpenShift AI Models-as-a-Service (MaaS) serves every model from a single OpenAI-compatible endpoint. This section builds one client from the MaaS base URL and API key; the embedding model reuses it.

**Prerequisites:**
- `MAAS_API_KEY`: Your authentication key for the MaaS API
- `MAAS_BASE_URL`: The complete OpenAI-compatible endpoint URL, used verbatim (e.g. `https://<host>/v1`)

&#x1F4A1; **Tip**: In OpenShift AI Workbench, you can add these as environment variables or data connections to avoid entering them manually each time.

In [ ]:
from ai4rag.utils.clients.maas_client import create_maas_client

MAAS_API_KEY = os.getenv("MAAS_API_KEY") or getpass.getpass("Please enter 'MAAS_API_KEY': ")
MAAS_BASE_URL = os.getenv("MAAS_BASE_URL") or getpass.getpass("Please enter 'MAAS_BASE_URL': ")

client = create_maas_client(base_url=MAAS_BASE_URL, api_key=MAAS_API_KEY)

### Prepare chunker

The chunker splits extracted documents into smaller chunks for more effective retrieval. Configuration includes:
- **Chunking Method**: The algorithm used to split text (e.g., recursive character splitting)
- **Chunk Size**: Maximum number of characters per chunk
- **Chunk Overlap**: Number of overlapping characters between consecutive chunks to preserve context

Proper chunking ensures that retrieved context is both relevant and fits within the model's context window.

In [ ]:
from ai4rag.rag.chunking import LangChainChunker, DoclingChunker

chunking_method = "{CHUNKING_METHOD}"
chunk_size = {CHUNK_SIZE}
chunk_overlap = {CHUNK_OVERLAP}

if chunking_method == "hybrid":
    chunker = DoclingChunker(max_tokens=chunk_size)
else:
    chunker = LangChainChunker(method=chunking_method, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

## Initialize the graph store

Use the same collection name in the inference notebook. Neo4j graph retrieval is collection-isolated and supports `search_mode="graph"` only.

In [ ]:
from ai4rag.rag.embedding.openai_model import OpenAIEmbeddingModel, OpenAIEmbeddingParams
from ai4rag.rag.foundation_models.openai_model import OpenAIFoundationModel
from ai4rag.rag.vector_store.config import Neo4jConfig
from ai4rag.rag.vector_store.neo4j import Neo4jGraphStore

embedding_model = OpenAIEmbeddingModel(
    client=client,
    model_id="{EMBEDDING_MODEL_ID}",
    params=OpenAIEmbeddingParams(**{EMBEDDING_PARAMS}),
)
foundation_model = OpenAIFoundationModel(
    client=client,
    model_id="{FM_MODEL_ID}",
    params={FM_PARAMS},
)
collection_name = "{COLLECTION_NAME}"
vector_store = Neo4jGraphStore(
    embedding_model=embedding_model,
    config=Neo4jConfig.from_env(),
    collection_name=collection_name,
    foundation_model=foundation_model,
    kg_extraction_config={KG_EXTRACTION_CONFIG},
)

### Upload chunks to the knowledge graph

Each chunk is embedded, stored, and used to extract entities and relationships in the selected Neo4j collection.

In [ ]:
from ai4rag.utils.docling_io import load_docling_documents

documents = load_docling_documents(extracted_text_dir)

for document in documents:
    chunked_documents = chunker.split_documents([document])
    vector_store.add_documents(chunked_documents)

### Retrieve graph contexts for a sample question

In [ ]:
from dataclasses import asdict
from pprint import pprint

sample_question = input()
results = vector_store.search(
    query=sample_question,
    k=5,
    search_mode="graph",
    **{GRAPH_RETRIEVAL_CONFIG},
)
for chunk in results:
    pprint(asdict(chunk), indent=4)

---

## Summary

This notebook used the MaaS indexing workflow to discover and extract documents, then built a Neo4j knowledge graph that is ready for graph retrieval.

In [ ]:
vector_store.close()